# 🏎️ LazyPredict Benchmarking
**Extended · Data Pattern Recognition for the Rest of Us**

> Before committing to a model, benchmark dozens of algorithms in one call. LazyPredict tells you which model families are worth deeper investment — in minutes instead of days.

### Dataset
**Telecom Customer Churn**
Predict which customers will cancel their service. Churn costs telecom companies billions annually — retaining customers is 5-25x cheaper than acquiring new ones. This is a classic imbalanced binary classification problem with real business stakes.

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install lazypredict -q
from lazypredict.Supervised import LazyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print("\u2713 Setup complete — lazypredict installed")

In [ ]:
# Telecom Churn Dataset (IBM Telco — widely used in industry)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    telco = pd.read_csv(url)
    telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')
    telco = telco.dropna()
    telco['Churn_num'] = (telco['Churn'] == 'Yes').astype(int)
    cat_cols = telco.select_dtypes('object').columns.drop(['customerID','Churn'])
    le = LabelEncoder()
    for c in cat_cols:
        telco[c] = le.fit_transform(telco[c])
    feature_cols = [c for c in telco.columns
                    if c not in ['customerID','Churn','Churn_num']]
    X = telco[feature_cols].values
    y = telco['Churn_num'].values
    print("Loaded IBM Telco Churn dataset")
except:
    np.random.seed(42)
    n = 7043  # matches real Telco dataset size
    tenure         = np.random.randint(0, 72, n)
    monthly_charges= np.random.uniform(18, 120, n)
    total_charges  = tenure * monthly_charges * np.random.uniform(0.9,1.1,n)
    contract       = np.random.choice([0,1,2], n, p=[0.55,0.24,0.21])  # M2M/1yr/2yr
    internet_service=np.random.choice([0,1,2], n, p=[0.21,0.44,0.35])  # No/DSL/Fiber
    tech_support   = np.random.binomial(1,0.28,n)
    paperless      = np.random.binomial(1,0.59,n)
    num_services   = np.random.randint(0,7,n)
    senior         = np.random.binomial(1,0.16,n)

    log_odds = (-1.5
                - 0.05*tenure
                + 0.015*monthly_charges
                + 1.2*(contract==0).astype(float)
                - 0.8*(contract==2).astype(float)
                + 0.4*(internet_service==2).astype(float)
                - 0.3*tech_support
                + 0.2*senior)
    prob_churn = 1/(1+np.exp(-log_odds))
    churn = (np.random.uniform(0,1,n) < prob_churn).astype(int)

    X = np.column_stack([tenure,monthly_charges,total_charges,contract,
                         internet_service,tech_support,paperless,
                         num_services,senior])
    y = churn
    feature_cols = ['tenure','monthly_charges','total_charges','contract',
                    'internet_service','tech_support','paperless',
                    'num_services','senior']
    print("Using synthetic Telco-like churn dataset")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
print(f"\nTelecom Churn dataset: {len(y):,} customers")
print(f"Churn rate: {y.mean():.1%}")
print(f"Training: {len(y_tr):,}  |  Test: {len(y_te):,}")
print("\nRunning LazyPredict on all sklearn classifiers...")
print("(This benchmarks 30+ models — may take 2-3 minutes)")

In [ ]:
# Run LazyPredict — benchmark all classifiers
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_tr, X_te, y_tr, y_te)

print("=== LazyPredict Results — Telecom Churn (sorted by AUC-ROC) ===\n")
# Show top 15
display_cols = [c for c in ['Accuracy','Balanced Accuracy','ROC AUC','F1 Score','Time Taken']
                if c in models_df.columns]
if 'ROC AUC' in models_df.columns:
    models_sorted = models_df.sort_values('ROC AUC', ascending=False)
else:
    models_sorted = models_df.sort_values('Accuracy', ascending=False)
print(models_sorted.head(15)[display_cols].to_string())

In [ ]:
# Visualize the benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# AUC-ROC comparison
metric = 'ROC AUC' if 'ROC AUC' in models_df.columns else 'Accuracy'
top_models = models_sorted.head(15)
colors_bar = ['#1a7a45' if i < 3 else '#1e5fa8' if i < 8 else '#888'
              for i in range(len(top_models))]
axes[0].barh(range(len(top_models)), top_models[metric].values[::-1],
             color=colors_bar[::-1], edgecolor='white')
axes[0].set_yticks(range(len(top_models)))
axes[0].set_yticklabels([n[:25] for n in top_models.index[::-1]], fontsize=9)
axes[0].set_xlabel(metric)
axes[0].set_title(f'LazyPredict: {metric} by Model\n'
                  f'Green = top 3, Blue = top 8')
axes[0].axvline(top_models[metric].median(), color='#e85d20', lw=1.5, ls='--',
               label=f'Median = {top_models[metric].median():.3f}')
axes[0].legend()

# Speed vs accuracy tradeoff
if 'Time Taken' in models_df.columns:
    axes[1].scatter(models_df['Time Taken'], models_df[metric],
                    color='#1e5fa8', alpha=0.7, s=40)
    for i, (name, row) in enumerate(models_df.iterrows()):
        if row[metric] > models_df[metric].quantile(0.85):
            axes[1].annotate(name[:15], (row['Time Taken'], row[metric]),
                            fontsize=7, alpha=0.8)
    axes[1].set_xlabel('Training Time (seconds)')
    axes[1].set_ylabel(metric)
    axes[1].set_title('Speed vs Performance Tradeoff\n'
                      'Best = top-left (fast AND accurate)')

plt.tight_layout(); plt.show()
best_model = models_sorted.index[0]
print(f"\n\u2714 LazyPredict winner: {best_model}")
print(f"   This is a STARTING POINT — now tune this model with proper cross-validation")
print(f"   LazyPredict does NOT replace rigorous model evaluation!")

In [ ]:
# Business framing: what does AUC mean for churn?
print("=== Business Interpretation ===\n")
print("A churn model with AUC=0.85 means:")
print("  If you randomly pick 1 churner and 1 non-churner,")
print("  the model ranks the churner higher 85% of the time.")
print()
print("At a retention budget of $50/customer and 7,043 customers:")
churn_rate = y.mean()
n_churners = int(len(y) * churn_rate)
budget_per_customer = 50
print(f"  Total churners to prevent: ~{n_churners}")
print(f"  If model catches 70% of churners at a 2:1 false positive rate:")
caught = int(n_churners * 0.70)
contacted = caught * 3  # 2:1 FP ratio
cost = contacted * budget_per_customer
prevented_revenue = caught * 120 * 12  # $120/mo avg, 1 year
print(f"  Customers contacted: {contacted:,}")
print(f"  Campaign cost: ${cost:,}")
print(f"  Revenue retained (est.): ${prevented_revenue:,}")
print(f"  Net ROI: {(prevented_revenue - cost)/cost*100:.0f}%")

In [ ]:
# @title 📝 Quiz — LazyPredict Benchmarking
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is LazyPredict's primary use case and what is it NOT good for?
# @markdown **Q2:** Why is accuracy a poor metric for churn prediction (hint: class imbalance)?
# @markdown **Q3:** The top LazyPredict model uses default hyperparameters. What should you do next?
# @markdown **Q4:** How would you explain AUC-ROC to a non-technical marketing manager?
# @markdown **Q5:** For churn: would you rather minimize false positives or false negatives? Why?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "LazyPredict Benchmarking"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*